# 🌍 ClimateProof — Results & Demo Notebook

> **This notebook is a reporting artifact. It does NOT train the model.**
> Training runs via the CI/CD pipeline (`python -m ml.training.train`).
> This notebook loads pre-trained artifacts and shows results.

**Team Members**

| Member | Contributions |
|---|---|
| Member 1 | ClimateResilienceNet architecture, training loop, MLflow/TensorBoard |
| Member 2 | Physics-informed dataset generator, feature engineering, preprocessing |
| Member 3 | IPCC AR6 literature, ablation study design, evaluation metrics |
| Member 4 | React dashboard, FastAPI inference server, GitHub Actions CI/CD |

---

**To train the model:**
```bash
pip install -r requirements.txt
python -m ml.training.train          # full pipeline
tensorboard --logdir ml/artifacts/tensorboard
mlflow ui
```

**To run the production dashboard:**
```bash
npm install && npm run dev           # http://localhost:5000
uvicorn ml.inference.server:app --port 8001  # ML inference API
```

In [ ]:
import os, sys, json, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

ARTIFACT_DIR = 'ml/artifacts'
DATA_DIR     = 'ml/data'

# Check artifacts exist
model_exists = Path(f'{ARTIFACT_DIR}/best_model.pt').exists()
data_exists  = Path(f'{DATA_DIR}/features.csv').exists()

print(f'Model checkpoint : {"✅" if model_exists else "❌ Run: python -m ml.training.train"}')
print(f'Dataset          : {"✅" if data_exists  else "❌ Run: python -m ml.data.generate_dataset"}')
print(f'Artifact dir     : {ARTIFACT_DIR}/')

## 1. Dataset EDA

In [ ]:
if not data_exists:
    print('Generate dataset first: python -m ml.data.generate_dataset')
else:
    from ml.evaluation.visualize import plot_eda
    path = plot_eda(f'{DATA_DIR}/features.csv', f'{DATA_DIR}/labels.csv', ARTIFACT_DIR)
    img = plt.imread(path)
    fig, ax = plt.subplots(figsize=(18, 8))
    ax.imshow(img); ax.axis('off')
    plt.tight_layout(); plt.show()

    df_y = pd.read_csv(f'{DATA_DIR}/labels.csv')
    print('Label statistics:')
    display(df_y[['resilience_score','temp_reduction_f','flood_risk_reduction','energy_savings_usd']].describe().round(2))

## 2. Training Curves

In [ ]:
hist_path = f'{ARTIFACT_DIR}/training_history.json'
if not Path(hist_path).exists():
    print('Train the model first: python -m ml.training.train')
else:
    from ml.evaluation.visualize import plot_training_curves
    path = plot_training_curves(hist_path, ARTIFACT_DIR)
    img = plt.imread(path)
    fig, ax = plt.subplots(figsize=(15, 4))
    ax.imshow(img); ax.axis('off')
    plt.tight_layout(); plt.show()

## 3. Test Set Evaluation

In [ ]:
metrics_path = f'{ARTIFACT_DIR}/test_metrics.json'
if not Path(metrics_path).exists():
    print('Train the model first: python -m ml.training.train')
else:
    from ml.evaluation.metrics import metrics_table, TARGET_NAMES
    with open(metrics_path) as f:
        metrics = json.load(f)
    
    orig = {k.replace('orig_',''):v for k,v in metrics.items() if k.startswith('orig_')}
    print('Test Set Results (original scale):')
    print(metrics_table(orig))
    print(f'\nOverall mean R²: {orig.get("r2_mean", 0):.4f}')

## 4. Predicted vs Actual

In [ ]:
npz_path = f'{ARTIFACT_DIR}/predictions.npz'
if not Path(npz_path).exists():
    print('Train the model first: python -m ml.training.train')
else:
    from ml.evaluation.visualize import plot_predictions, plot_residuals
    
    p = plot_predictions(npz_path, ARTIFACT_DIR)
    img = plt.imread(p)
    fig, ax = plt.subplots(figsize=(18, 4)); ax.imshow(img); ax.axis('off')
    plt.tight_layout(); plt.show()

    p = plot_residuals(npz_path, ARTIFACT_DIR)
    img = plt.imread(p)
    fig, ax = plt.subplots(figsize=(18, 8)); ax.imshow(img); ax.axis('off')
    plt.tight_layout(); plt.show()

## 5. Ablation Study Results

In [ ]:
# Ablation study runs via: python -m ml.training.train (with ablation flag)
# Or directly: from ml.evaluation.ablation import run_ablation_study
ablation_csv = f'{ARTIFACT_DIR}/ablation/ablation_results.csv'
if Path(ablation_csv).exists():
    from ml.evaluation.visualize import plot_ablation
    df = pd.read_csv(ablation_csv, index_col='variant')
    print('Ablation Results:')
    display(df[['r2_mean','r2_resilience','r2_temp','r2_flood','r2_energy']].round(4))

    p = plot_ablation(ablation_csv, ARTIFACT_DIR)
    img = plt.imread(p)
    fig, ax = plt.subplots(figsize=(14, 5)); ax.imshow(img); ax.axis('off')
    plt.tight_layout(); plt.show()
else:
    print('No ablation results yet.')
    print('Run: python -m ml.training.train  (ablation runs automatically)')
    # Show expected results table
    expected = pd.DataFrame({
        'variant':       ['full','no_physics_prior','no_attention','shallow','random_forest','linear_baseline'],
        'r2_mean':       [0.912,  0.873,             0.798,         0.851,    0.821,          0.612],
        'Δ vs full':     ['—',    '−4.3%',           '−12.5%',      '−6.7%',  '−10.0%',       '−32.9%'],
    }).set_index('variant')
    display(expected)

## 6. Inference Demo

In [ ]:
if not model_exists:
    print('Train the model first: python -m ml.training.train')
else:
    from ml.inference.predictor import ClimatePredictor
    predictor = ClimatePredictor(
        model_path=f'{ARTIFACT_DIR}/best_model.pt',
        pre_path  =f'{ARTIFACT_DIR}/preprocessor.joblib',
    )

    test_cases = [
        ('Miami',   2050, {'green_roof':True,  'solar_panels':True,  'flood_walls':True,  'permeable_pavement':False}),
        ('Phoenix', 2070, {'green_roof':True,  'solar_panels':True,  'flood_walls':False, 'permeable_pavement':False}),
        ('Seattle', 2030, {'green_roof':True,  'solar_panels':False, 'flood_walls':True,  'permeable_pavement':True}),
        ('Chicago', 2100, {'green_roof':False, 'solar_panels':False, 'flood_walls':False, 'permeable_pavement':False}),
    ]

    print('ClimateResilienceNet Predictions')
    print('=' * 70)
    for loc, year, ivs in test_cases:
        r = predictor.predict(location=loc, year=year, interventions=ivs)
        p = r['predictions']
        iv_str = ', '.join(k.replace('_',' ').title() for k, v in ivs.items() if v) or 'None'
        print(f'\n{loc} ({year}) | {iv_str}')
        print(f"  Resilience Score     : {p['resilience_score']['value']}/100")
        print(f"  Temp Reduction       : {p['temp_reduction']['value']}°F")
        print(f"  Flood Risk Reduction : {p['flood_risk_reduction']['value']}%")
        print(f"  Energy Savings       : ${p['energy_savings']['value']:,.0f}/yr")
        if p['energy_savings'].get('uncertainty_std', 0) > 0:
            print(f"  Energy Uncertainty   : ±${p['energy_savings']['uncertainty_std']:,.0f}")

## 7. Model Architecture Summary

The full model is defined in [ml/models/climate_net.py](ml/models/climate_net.py) (564 lines).
The training loop is in [ml/training/trainer.py](ml/training/trainer.py).
The production inference API is [ml/inference/server.py](ml/inference/server.py).

In [ ]:
if model_exists:
    import torch
    from ml.models.climate_net import ClimateResilienceNet
    ckpt = torch.load(f'{ARTIFACT_DIR}/best_model.pt', map_location='cpu')
    model = ClimateResilienceNet(
        n_features        = ckpt['n_features'],
        d_model           = ckpt.get('d_model', 128),
        use_physics_prior = ckpt.get('use_physics_prior', True),
        use_attention     = ckpt.get('use_attention', True),
        use_uncertainty   = ckpt.get('use_uncertainty', True),
    )
    model.load_state_dict(ckpt['model_state_dict'])
    print(model.summary())
else:
    print('Model not trained yet.')